# Algeria Trade Master Table Builder
## Data Collection & Preparation — Phase 2

This notebook builds the final master table from the yearly merged parquet files
produced by `data_pipeline.ipynb`. It runs on **Kaggle** (free tier, 30GB RAM)
to overcome local storage and memory limitations.

### Context
The raw BACI HS92 dataset (1995–2024) contains ~300 million rows across 30 yearly CSV files,
totaling ~4GB on disk and ~15GB in RAM when fully loaded — exceeding local machine capacity.
To solve this, the data pipeline notebook processed and merged each year individually on the
local machine, producing 30 lightweight parquet files (~80–500MB each) stored in `merged_by_year/`.
These were then zipped and uploaded to Kaggle as a dataset for final assembly here.

### What this notebook does
1. Reads all yearly merged parquets from the Kaggle dataset
2. Processes them in batches of 5 years to stay within RAM limits
3. Writes the final master table incrementally using PyArrow (no full RAM concat)
4. Outputs `algeria_trade_master.parquet` (**Google Drive**: [algeria_trade_master.parquet](https://drive.google.com/file/d/17T00nZEu8RCmxQtG58B1erXdpo12pkYB/view?usp=sharing))— the clean, unified dataset ready for feature engineering

### Environment
- **Platform**: Kaggle Notebooks (free tier)
- **RAM**: 30GB available
- **Input dataset**: `loubnabensaoula/merged-by-year` (uploaded from local machine)
- link from google drive : [merged_by_year.zip](https://drive.google.com/file/d/1vjt-KpGDJDXOXcPlQpiaQcEip1yeYn_h/view?usp=sharing)
- **Output**: `algeria_trade_master.parquet` (~2.79GB, ~150M rows, 31 columns)

### Data Coverage
- **Years**: 2010–2024 (15 years)
- **Skipped**: 2019 — file was corrupted during local merge step (disk ran out mid-write)
- **Dropped columns**: `interest_rate_i`, `interest_rate_j` — 63% missing values, not reliable
- **Countries**: 200+
- **Products**: 5,000+ HS92 6-digit codes

### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| t | int16 | Year |
| k | string | HS92 product code (6-digit) |
| v | float32 | Trade value (FOB, thousands USD) |
| q | float32 | Quantity (tons) |
| iso3_i | string | Exporter ISO3 code |
| iso3_j | string | Importer ISO3 code |
| product_description | string | HS92 product description |
| continent_i/j | string | Continent of exporter/importer |
| landlocked_i/j | int8 | 1 if landlocked, 0 otherwise |
| lat_i/j, lon_i/j | float32 | Coordinates of exporter/importer |
| dist | float32 | Bilateral distance (km) |
| contig | int8 | 1 if share a border |
| comlang_off | int8 | 1 if share official language |
| colony | int8 | 1 if colonial link exists |
| gdp_i/j | float32 | GDP in current USD |
| gdp_growth_i/j | float32 | GDP growth rate (%) |
| population_i/j | float32 | Total population |
| inflation_i/j | float32 | Inflation rate (%) |
| trade_percent_gdp_i/j | float32 | Trade as % of GDP |
| unemployment_i/j | float32 | Unemployment rate (%) |

## Step 1 — Batch Creation (5 years per batch)

Loading all 15 yearly parquet files at once would require ~10GB RAM.
Instead we process them in **batches of 5 years**, concat each batch,
and save it as a temporary parquet file before moving to the next.

### Challenges handled
- **Corrupted file**: `merged_2019.parquet` was corrupted because local disk ran
  out of space mid-write during the merge step. It is skipped via the `SKIP` list.
- **Schema inconsistencies**: Early years (1995–1999) were merged with slightly
  different column names (`trade_pct_gdp` vs `trade_percent_gdp`) and contained
  extra numeric `i`, `j` columns. These are normalized in `clean_df()`.
- **Memory**: Each batch is deleted from RAM immediately after saving using `del` + `gc.collect()`.
- **Dropped columns**: `interest_rate_i` and `interest_rate_j` are dropped — they had
  over 63% missing values across all countries and years, making them unreliable for modeling.
- **Dtype optimization**: All float64 → float32, integers → int16/Int8 to minimize memory footprint.

### Environment path detection
Paths are auto-detected so this notebook runs identically on Kaggle and locally in VSCode.

In [ ]:
import os

# ── Environment Detection ────────────────────────────────────────────
if os.path.exists("/kaggle/working"):
    # Kaggle environment
    MERGE_DIR  = "/kaggle/input/datasets/loubnabensaoula/merged-by-year/merged_by_year"
    MASTER_OUT = "/kaggle/working/algeria_trade_master.parquet"
else:
    # Local VSCode environment
    ROOT       = os.path.abspath(os.path.join(os.getcwd(), ".."))
    MERGE_DIR  = os.path.join(ROOT, "data", "processed", "merged_by_year")
    MASTER_OUT = os.path.join(ROOT, "data", "processed", "algeria_trade_master.parquet")

print(f"Environment : {'Kaggle' if os.path.exists('/kaggle/working') else 'Local'}")
print(f"MERGE_DIR   : {MERGE_DIR}")
print(f"MASTER_OUT  : {MASTER_OUT}")

In [ ]:
import pandas as pd
import glob
import os
import gc

SKIP       = ["merged_2019.parquet"]
DROP_COLS  = ["interest_rate_i", "interest_rate_j"]

def clean_df(df):
    df["k"] = df["k"].astype(str)
    for col in ["landlocked_i","landlocked_j","contig","comlang_off","colony"]:
        if col in df.columns:
            df[col] = df[col].astype("Int8")
    df["t"] = df["t"].astype("int16")
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")
    for col in DROP_COLS:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    return df

# only 2010-2024
parquet_files = sorted(glob.glob(os.path.join(MERGE_DIR, "*.parquet")))
parquet_files = [
    f for f in parquet_files
    if os.path.basename(f) not in SKIP
    and int(os.path.basename(f).replace("merged_","").replace(".parquet","")) >= 2010
]
print(f"Files to process: {len(parquet_files)}")
for f in parquet_files:
    print(f"  {os.path.basename(f)}")

BATCH_SIZE = 5
batch_num  = 0

for i in range(0, len(parquet_files), BATCH_SIZE):
    batch = parquet_files[i:i+BATCH_SIZE]
    print(f"\nBatch {batch_num+1} — {[os.path.basename(f) for f in batch]}")
    dfs = []
    for path in batch:
        try:
            df = pd.read_parquet(path)
            df = clean_df(df)
            dfs.append(df)
            print(f"  loaded {os.path.basename(path)} — {df.shape}")
            del df
            gc.collect()
        except Exception as e:
            print(f"  SKIPPED {os.path.basename(path)} — {e}")
            continue
    if not dfs:
        batch_num += 1
        continue
    batch_df  = pd.concat(dfs, ignore_index=True)
    temp_path = f"/kaggle/working/batch_{batch_num}.parquet"
    batch_df.to_parquet(temp_path, index=False, compression="snappy")
    print(f"  batch saved — shape {batch_df.shape} — {os.path.getsize(temp_path)/1e6:.1f} MB")
    del dfs, batch_df
    gc.collect()
    batch_num += 1

print(f"\nTotal batches created: {batch_num}")

Files to process: 14
  merged_2010.parquet
  merged_2011.parquet
  merged_2012.parquet
  merged_2013.parquet
  merged_2014.parquet
  merged_2015.parquet
  merged_2016.parquet
  merged_2017.parquet
  merged_2018.parquet
  merged_2020.parquet
  merged_2021.parquet
  merged_2022.parquet
  merged_2023.parquet
  merged_2024.parquet

Batch 1 — ['merged_2010.parquet', 'merged_2011.parquet', 'merged_2012.parquet', 'merged_2013.parquet', 'merged_2014.parquet']
  loaded merged_2010.parquet — (9470765, 31)
  loaded merged_2011.parquet — (9640712, 31)
  loaded merged_2012.parquet — (9937050, 31)
  loaded merged_2013.parquet — (10098942, 31)
  loaded merged_2014.parquet — (10145672, 31)
  batch saved — shape (49293141, 31) — 499.8 MB

Batch 2 — ['merged_2015.parquet', 'merged_2016.parquet', 'merged_2017.parquet', 'merged_2018.parquet', 'merged_2020.parquet']
  loaded merged_2015.parquet — (10499686, 31)
  loaded merged_2016.parquet — (10569571, 31)
  loaded merged_2017.parquet — (10821738, 31)
  lo

## Step 2 — Incremental Master Table Assembly

The 3 batch files (~500MB each) cannot be concatenated in RAM using `pd.concat()`
— this would require loading ~1.5GB+ and crashed in previous attempts.

Instead we use **PyArrow's `ParquetWriter`** to write each batch directly to disk
one at a time, never holding more than one batch in memory simultaneously.

### Challenges handled
- **Schema mismatch between batches**: batch_0 (years 1995–1999) had a different
  column order and extra columns vs batch_1+. Fixed by:
  - Dropping extra columns (`i`, `j`, `trade_pct_gdp_i/j`)
  - Deduplicating columns with `df.loc[:, ~df.columns.duplicated()]`
  - Casting mismatched schemas with `table.cast(writer.schema)`
- **No RAM concat**: PyArrow writes row groups directly to the parquet file on disk,
  meaning peak RAM usage is only one batch (~2–3GB) at a time.
- **Final output**: `algeria_trade_master.parquet` — 2.79GB, ~150M rows, 31 columns.
  Uploaded to Google Drive for sharing with the team since it exceeds GitHub's 100MB limit.

### How to access the master table
- **Kaggle**: available as output of this notebook after commit
- **Google Drive**: [algeria_trade_master.parquet](https://drive.google.com/file/d/17T00nZEu8RCmxQtG58B1erXdpo12pkYB/view?usp=sharing)
- **Locally**: place in `data/processed/` and update paths in config cell

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import os
import gc


if os.path.exists(MASTER_OUT):
    os.remove(MASTER_OUT)

FINAL_COLS = [
    "t", "k", "v", "q", "iso3_i", "iso3_j", "product_description",
    "continent_i", "continent_j", "landlocked_i", "landlocked_j",
    "lat_i", "lon_i", "lat_j", "lon_j",
    "dist", "contig", "comlang_off", "colony",
    "gdp_i", "gdp_j", "gdp_growth_i", "gdp_growth_j",
    "population_i", "population_j", "inflation_i", "inflation_j",
    "trade_percent_gdp_i", "trade_percent_gdp_j",
    "unemployment_i", "unemployment_j"
]

batch_files = sorted(glob.glob("/kaggle/working/batch_*.parquet"))
print(f"Batch files found: {len(batch_files)}")

writer = None
for path in batch_files:
    print(f"  appending {os.path.basename(path)}...", end=" ")
    df = pd.read_parquet(path)
    
    for col in ["i", "j"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    if "trade_pct_gdp_i" in df.columns and "trade_percent_gdp_i" not in df.columns:
        df.rename(columns={"trade_pct_gdp_i":"trade_percent_gdp_i",
                           "trade_pct_gdp_j":"trade_percent_gdp_j"}, inplace=True)
    for col in ["trade_pct_gdp_i","trade_pct_gdp_j"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)
    df = df.loc[:, ~df.columns.duplicated()]
    cols = [c for c in FINAL_COLS if c in df.columns]
    df = df[cols]
    
    table = pa.Table.from_pandas(df, preserve_index=False)
    del df
    gc.collect()
    
    if writer is None:
        writer = pq.ParquetWriter(MASTER_OUT, table.schema, compression="snappy")
    try:
        writer.write_table(table)
    except:
        writer.write_table(table.cast(writer.schema))
    print("done")
    del table
    gc.collect()

writer.close()
print(f"\nSaved — {os.path.getsize(MASTER_OUT)/1e9:.2f} GB")
print("Commit now and download from Output tab!")

Batch files found: 3
  appending batch_0.parquet... done
  appending batch_1.parquet... done
  appending batch_2.parquet... done

Saved — 1.48 GB
Commit now and download from Output tab!
